In [22]:
import numpy as np
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [11]:
sys.path.append(os.path.join(os.getcwd(),'..','src'))

In [14]:
from decision_trees import DecisionTree
from utils import *

In [18]:
X_train = np.load('../Data/processed/bc_X_train.npy')
X_test = np.load('../Data/processed/bc_X_test.npy')
y_train = np.load('../Data/processed/bc_y_train.npy')
y_test = np.load('../Data/processed/bc_y_test.npy') 
feature_names = [f'feature{i}' for i in range(X_train.shape[1])]

print("loaded shapes")
print("X_train :",X_train.shape)
print("y_train :" , y_train.shape)
print("X_test :",X_test.shape) 
print("y_test :",y_test.shape)

loaded shapes
X_train : (455, 30)
y_train : (455,)
X_test : (114, 30)
y_test : (114,)


In [19]:
model_gini = DecisionTree(criterion='gini', max_depth=5, task='classification')
model_gini.fit(X_train, y_train)
pred_gini = model_gini.predict(X_test)
print('\nGini metrics:')
print('accuracy :', accuracy(y_test, pred_gini))
print('precision:', precision(y_test, pred_gini))
print('recall :', recall(y_test, pred_gini))
print('f1  :', f1_score(y_test, pred_gini))



Gini metrics:
accuracy : 0.9385964912280702
precision: 0.9090909090909091
recall : 0.9302325581395349
f1  : 0.9195402298850575


In [20]:
model_entropy = DecisionTree(criterion='entropy', max_depth=5, task='classification')
model_entropy.fit(X_train, y_train)
pred_entropy = model_entropy.predict(X_test)
print('\nEntropy metrics:')
print('accuracy :', accuracy(y_test, pred_entropy))
print('precision:', precision(y_test, pred_entropy))
print('recall :', recall(y_test, pred_entropy))
print('f1   :', f1_score(y_test, pred_entropy))


Entropy metrics:
accuracy : 0.9385964912280702
precision: 0.95
recall : 0.8837209302325582
f1   : 0.9156626506024096


In [23]:
results = pd.DataFrame([
    ['gini', accuracy(y_test, pred_gini), precision(y_test, pred_gini), recall(y_test, pred_gini), f1_score(y_test, pred_gini)],
    ['entropy', accuracy(y_test, pred_entropy), precision(y_test, pred_entropy), recall(y_test, pred_entropy), f1_score(y_test, pred_entropy)],
], columns=['criterion', 'accuracy', 'precision', 'recall', 'f1'])

In [24]:
print('\nComparison table:')
print(results.to_markdown(index=False))


Comparison table:
| criterion   |   accuracy |   precision |   recall |       f1 |
|:------------|-----------:|------------:|---------:|---------:|
| gini        |   0.938596 |    0.909091 | 0.930233 | 0.91954  |
| entropy     |   0.938596 |    0.95     | 0.883721 | 0.915663 |


In [25]:
best_model = model_gini if accuracy(y_test, pred_gini) >= accuracy(y_test, pred_entropy) else model_entropy
best_pred = pred_gini if best_model is model_gini else pred_entropy
best_name = 'gini' if best_model is model_gini else 'entropy'

In [26]:
print('\nBetter model:', best_name)
print('Why: it has the higher or equal test accuracy on this split; if metrics tie, choose the one with the better precision/recall balance.')


Better model: gini
Why: it has the higher or equal test accuracy on this split; if metrics tie, choose the one with the better precision/recall balance.


In [27]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 7, 8, 10]:
    m = DecisionTree(criterion='gini', max_depth=d, task='classification')
    m.fit(X_train, y_train)
    train_pred = m.predict(X_train)
    test_pred = m.predict(X_test)
    rows.append({
        'depth': d,
        'train_accuracy': accuracy(y_train, train_pred),
        'test_accuracy': accuracy(y_test, test_pred)
    })

In [28]:
df = pd.DataFrame(rows)
print('\nDepth study:')
print(df.to_markdown(index=False))


Depth study:
|   depth |   train_accuracy |   test_accuracy |
|--------:|-----------------:|----------------:|
|       1 |         0.920879 |        0.894737 |
|       2 |         0.92967  |        0.929825 |
|       3 |         0.978022 |        0.938596 |
|       4 |         0.995604 |        0.938596 |
|       5 |         0.995604 |        0.938596 |
|       6 |         0.997802 |        0.938596 |
|       7 |         1        |        0.938596 |
|       8 |         1        |        0.938596 |
|      10 |         1        |        0.938596 |


In [31]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 7, 8, 10]:
    m = DecisionTree(criterion='gini', max_depth=d, task='classification')
    m.fit(X_train, y_train)
    train_pred = m.predict(X_train)
    test_pred = m.predict(X_test)
    rows.append({
        'depth': d,
        'train_accuracy': accuracy(y_train, train_pred),
        'test_accuracy': accuracy(y_test, test_pred)
    })

df = pd.DataFrame(rows)
print('\nDepth study:')
print(df.to_markdown(index=False))

plt.figure(figsize=(7, 4))
plt.plot(df['depth'], df['train_accuracy'], marker='o', label='Train')
plt.plot(df['depth'], df['test_accuracy'], marker='o', label='Test')
plt.xlabel('Max depth')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('depth_accuracy.png', dpi=160)
plt.close()



Depth study:
|   depth |   train_accuracy |   test_accuracy |
|--------:|-----------------:|----------------:|
|       1 |         0.920879 |        0.894737 |
|       2 |         0.92967  |        0.929825 |
|       3 |         0.978022 |        0.938596 |
|       4 |         0.995604 |        0.938596 |
|       5 |         0.995604 |        0.938596 |
|       6 |         0.997802 |        0.938596 |
|       7 |         1        |        0.938596 |
|       8 |         1        |        0.938596 |
|      10 |         1        |        0.938596 |


In [32]:
print('\nBest model tree:')
print_tree(best_model.root, feature_names=feature_names) 


Best model tree:
If feature7 < 0.05128:
  If feature20 < 16.83:
    If feature10 < 0.62555:
      If feature24 < 0.17765:
        If feature14 < 0.003309:
          Leaf: value = 0
        Else:
          Leaf: value = 0
      Else:
        Leaf: value = 1
    Else:
      If feature4 < 0.09068:
        Leaf: value = 0
      Else:
        Leaf: value = 1
  Else:
    If feature1 < 16.189999999999998:
      Leaf: value = 0
    Else:
      If feature17 < 0.010125499999999999:
        Leaf: value = 1
      Else:
        Leaf: value = 0
Else:
  If feature27 < 0.14655:
    If feature22 < 115.25:
      If feature1 < 21.055:
        Leaf: value = 0
      Else:
        Leaf: value = 1
    Else:
      Leaf: value = 1
  Else:
    If feature16 < 0.13565:
      Leaf: value = 1
    Else:
      Leaf: value = 0


In [34]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Actual 0', 'Actual 1'])
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=160)
plt.close()

print('\nConfusion matrix:')
print(cm) 


Confusion matrix:
[[67  4]
 [ 3 40]]
